# NFL Player Contact Detection: Distance Baseline First Experiment

This notebook turns the starter notebook's tracking-distance idea into a reproducible first experiment. It tunes a player-player distance threshold with MCC, applies that threshold to the test sample submission rows, and writes `submission.csv`.


## 1. Setup and Configuration

The defaults are Kaggle Code Competition friendly: fixed input path, no internet dependency, grouped validation, and a single output file.


In [ ]:
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from IPython.display import display
from sklearn.metrics import matthews_corrcoef
from sklearn.model_selection import GroupShuffleSplit

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "legend.frameon": False,
})

NOTEBOOK_VERSION = "DISTANCE_BASELINE_V9_CANONICAL_PATH"
NOTEBOOK_NAME = "Distance Baseline"
RANDOM_STATE = 42
TARGET = "contact"
ID_COL = "contact_id"
RUN_FAST = False
FAST_SAMPLE_PLAYS = 25
DISTANCE_THRESHOLDS = np.round(np.arange(0.2, 3.05, 0.1), 2)

REQUIRED_DATA_FILES = [
    "train_labels.csv",
    "sample_submission.csv",
    "train_player_tracking.csv",
    "test_player_tracking.csv",
]

DATA_DIR = Path("/kaggle/input/competitions/nfl-player-contact-detection")
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")

missing_files = [
    filename for filename in REQUIRED_DATA_FILES
    if not (DATA_DIR / filename).exists()
]
if missing_files:
    raise FileNotFoundError(
        f"Missing required files in {DATA_DIR}: {missing_files}"
    )

print(f"NOTEBOOK_VERSION: {NOTEBOOK_VERSION}")
print(f"NOTEBOOK_NAME: {NOTEBOOK_NAME}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR.resolve()}")


### Version Check

When this notebook runs on Kaggle, the first code cell prints `NOTEBOOK_VERSION = DISTANCE_BASELINE_V9_CANONICAL_PATH`. If Kaggle shows an older version string, pull/sync the latest notebook before trusting the output.


## 2. Helper Functions


In [ ]:
def reduce_memory_usage(df: pd.DataFrame) -> pd.DataFrame:
    """Downcast dataframe columns to reduce notebook memory usage."""
    out = df.copy()
    for col in out.columns:
        dtype = out[col].dtype
        if pd.api.types.is_integer_dtype(dtype):
            out[col] = pd.to_numeric(out[col], downcast="integer")
        elif pd.api.types.is_float_dtype(dtype):
            out[col] = pd.to_numeric(out[col], downcast="float")
        elif pd.api.types.is_object_dtype(dtype):
            nunique = out[col].nunique(dropna=False)
            if nunique / max(len(out), 1) < 0.5 and col != ID_COL:
                out[col] = out[col].astype("category")
    return out


def parse_contact_id(df: pd.DataFrame) -> pd.DataFrame:
    """Split Kaggle contact_id into play, step, player1, and player2 fields."""
    out = df.copy()
    parts = out[ID_COL].astype(str).str.split("_", expand=True)
    out["game_play"] = parts[0] + "_" + parts[1]
    out["step"] = parts[2].astype("int16")
    out["nfl_player_id_1"] = parts[3].astype(str)
    out["nfl_player_id_2"] = parts[4].astype(str)
    out["contact_type"] = np.where(
        out["nfl_player_id_2"].eq("G"), "ground", "player_player"
    )
    return out


def dataframe_overview(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """Summarize column types, missingness, unique values, and memory."""
    return pd.DataFrame({
        "dataset": name,
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": df.isna().mean().mul(100),
        "unique": df.nunique(dropna=False),
        "memory_mb": df.memory_usage(deep=True).div(1024 ** 2),
    }).sort_values(["missing_pct", "unique"], ascending=[False, False])


def maybe_sample_plays(df: pd.DataFrame, n_plays: int) -> pd.DataFrame:
    """Return all rows for a reproducible sample of game_play values."""
    plays = pd.Series(df["game_play"].unique()).sample(
        min(n_plays, df["game_play"].nunique()),
        random_state=RANDOM_STATE,
    )
    return df[df["game_play"].isin(plays)].copy()


def create_football_field(
    linenumbers: bool = True,
    endzones: bool = True,
    figsize: tuple[int, int] = (12, 6),
    line_color: str = "black",
    field_color: str = "white",
    endzone_color=None,
    ax=None,
):
    """Draw a football field for tracking-data visualization."""
    if endzone_color is None:
        endzone_color = field_color
    if ax is None:
        _, ax = plt.subplots(figsize=figsize)

    rect = patches.Rectangle(
        (0, 0), 120, 53.3, linewidth=0.1, edgecolor=line_color,
        facecolor=field_color, zorder=0,
    )
    ax.add_patch(rect)

    for x in range(10, 120, 10):
        ax.plot([x, x], [0, 53.3], color=line_color, linewidth=0.8)
    if endzones:
        ax.add_patch(patches.Rectangle(
            (0, 0), 10, 53.3, linewidth=0.1, edgecolor=line_color,
            facecolor=endzone_color, alpha=0.5, zorder=0,
        ))
        ax.add_patch(patches.Rectangle(
            (110, 0), 10, 53.3, linewidth=0.1, edgecolor=line_color,
            facecolor=endzone_color, alpha=0.5, zorder=0,
        ))

    if linenumbers:
        for x in range(20, 110, 10):
            number = x if x <= 50 else 120 - x
            label = str(number - 10)
            ax.text(x, 5, label, ha="center", fontsize=16, color=line_color)
            ax.text(
                x, 48.3, label, ha="center", fontsize=16,
                color=line_color, rotation=180,
            )

    hash_range = range(11, 110) if endzones else range(1, 120)
    for x in hash_range:
        ax.plot([x, x], [0.4, 0.7], color=line_color, linewidth=0.6)
        ax.plot([x, x], [52.6, 52.9], color=line_color, linewidth=0.6)
        ax.plot([x, x], [22.91, 23.57], color=line_color, linewidth=0.6)
        ax.plot([x, x], [29.73, 30.39], color=line_color, linewidth=0.6)

    ax.set_xlim(-5, 125)
    ax.set_ylim(-5, 58.3)
    ax.axis("off")
    return ax


def compute_pair_distances(
    contacts: pd.DataFrame,
    tracking: pd.DataFrame,
) -> pd.DataFrame:
    """Attach player coordinates and compute player-player distance in yards."""
    pairs = contacts[contacts["nfl_player_id_2"].ne("G")].copy()
    track_cols = [
        "game_play", "step", "nfl_player_id", "x_position", "y_position"
    ]
    track = tracking[track_cols].copy()
    for col in ["nfl_player_id", "nfl_player_id_1", "nfl_player_id_2"]:
        if col in track.columns:
            track[col] = track[col].astype(str)
        if col in pairs.columns:
            pairs[col] = pairs[col].astype(str)

    out = pairs.merge(
        track,
        left_on=["game_play", "step", "nfl_player_id_1"],
        right_on=["game_play", "step", "nfl_player_id"],
        how="left",
    ).rename(columns={
        "x_position": "x_position_1",
        "y_position": "y_position_1",
    }).drop(columns=["nfl_player_id"])

    out = out.merge(
        track,
        left_on=["game_play", "step", "nfl_player_id_2"],
        right_on=["game_play", "step", "nfl_player_id"],
        how="left",
    ).rename(columns={
        "x_position": "x_position_2",
        "y_position": "y_position_2",
    }).drop(columns=["nfl_player_id"])

    out["distance"] = np.sqrt(
        np.square(out["x_position_1"] - out["x_position_2"])
        + np.square(out["y_position_1"] - out["y_position_2"])
    )
    return out


def score_distance_thresholds(
    df: pd.DataFrame,
    thresholds: np.ndarray,
) -> pd.DataFrame:
    """Score distance thresholds while predicting ground rows as no-contact."""
    rows = []
    y_true = df[TARGET].astype(int).to_numpy()
    contact_type = df["contact_type"].to_numpy()
    distances = df["distance"].to_numpy()
    is_pair = contact_type == "player_player"

    for threshold in thresholds:
        pred = np.zeros(len(df), dtype=np.int8)
        pred[is_pair] = np.nan_to_num(distances[is_pair], nan=np.inf) <= threshold
        rows.append({
            "threshold": threshold,
            "mcc": matthews_corrcoef(y_true, pred),
            "positive_rate": pred.mean(),
        })
    return pd.DataFrame(rows).sort_values("mcc", ascending=False)


## 3. Load Data


In [ ]:
labels = reduce_memory_usage(pd.read_csv(
    DATA_DIR / "train_labels.csv",
    parse_dates=["datetime"],
))
train_tracking = reduce_memory_usage(pd.read_csv(
    DATA_DIR / "train_player_tracking.csv",
    parse_dates=["datetime"],
))
test_tracking = reduce_memory_usage(pd.read_csv(
    DATA_DIR / "test_player_tracking.csv",
    parse_dates=["datetime"],
))
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

labels = parse_contact_id(labels)
sample_submission = parse_contact_id(sample_submission)

print(f"labels: {labels.shape}")
print(f"train_tracking: {train_tracking.shape}")
print(f"test_tracking: {test_tracking.shape}")
print(f"sample_submission: {sample_submission.shape}")
labels.head()


## 4. Validation Threshold Search

Use a grouped split by `game_play` so threshold tuning is evaluated on held-out plays. Ground contact rows are kept in scoring and predicted as zero, matching this baseline's test-time behavior.


In [ ]:
train_rows = labels[[
    ID_COL, TARGET, "game_play", "step", "nfl_player_id_1",
    "nfl_player_id_2", "contact_type",
]].copy()
if RUN_FAST:
    train_rows = maybe_sample_plays(train_rows, FAST_SAMPLE_PLAYS)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=RANDOM_STATE,
)
_, valid_idx = next(splitter.split(train_rows, groups=train_rows["game_play"]))
valid_rows = train_rows.iloc[valid_idx].copy()

valid_distances = compute_pair_distances(valid_rows, train_tracking)
valid_scoring = valid_rows.merge(
    valid_distances[[ID_COL, "distance"]],
    on=ID_COL,
    how="left",
)
threshold_scores = score_distance_thresholds(
    valid_scoring,
    DISTANCE_THRESHOLDS,
)
best_threshold = float(threshold_scores.iloc[0]["threshold"])

print(f"Validation rows: {len(valid_scoring):,}")
print(f"Validation plays: {valid_scoring['game_play'].nunique():,}")
print(f"Best threshold: {best_threshold:.2f} yards")
display(threshold_scores.head(12))

fig, ax = plt.subplots(figsize=(8, 4))
sns.lineplot(
    data=threshold_scores.sort_values("threshold"),
    x="threshold",
    y="mcc",
    marker="o",
    ax=ax,
)
ax.axvline(best_threshold, color="black", linestyle="--", linewidth=1)
ax.set_title("Validation MCC by distance threshold")
plt.show()


## 5. Full-Train Threshold Check

Score the chosen threshold on the labeled training table as a sanity check. This is not the validation estimate, but it helps confirm prediction rate and contact-type behavior before writing a submission.


In [ ]:
train_distances = compute_pair_distances(train_rows, train_tracking)
train_scoring = train_rows.merge(
    train_distances[[ID_COL, "distance"]],
    on=ID_COL,
    how="left",
)
train_pred = np.zeros(len(train_scoring), dtype=np.int8)
is_pair = train_scoring["contact_type"].eq("player_player").to_numpy()
train_pred[is_pair] = (
    np.nan_to_num(train_scoring.loc[is_pair, "distance"].to_numpy(), nan=np.inf)
    <= best_threshold
)

print(f"Full labeled MCC at chosen threshold: {matthews_corrcoef(train_scoring[TARGET], train_pred):.5f}")
print(f"Predicted positive rate: {train_pred.mean():.5f}")
display(pd.crosstab(
    train_scoring["contact_type"],
    train_scoring[TARGET],
    normalize="index",
).rename(columns={0: "label_0", 1: "label_1"}))

gc.collect()


## 6. Create Submission

Apply the selected threshold to player-player rows in `sample_submission.csv`. Ground rows remain zero for this baseline.


In [ ]:
test_rows = sample_submission[[
    ID_COL, "game_play", "step", "nfl_player_id_1",
    "nfl_player_id_2", "contact_type",
]].copy()
test_distances = compute_pair_distances(test_rows, test_tracking)
submission = sample_submission[[ID_COL]].copy()
submission[TARGET] = 0

pair_predictions = test_distances[[ID_COL, "distance"]].copy()
pair_predictions[TARGET] = (
    np.nan_to_num(pair_predictions["distance"].to_numpy(), nan=np.inf)
    <= best_threshold
).astype(np.int8)

submission = submission.merge(
    pair_predictions[[ID_COL, TARGET]].rename(columns={TARGET: "pair_contact"}),
    on=ID_COL,
    how="left",
)
submission[TARGET] = submission["pair_contact"].fillna(0).astype(np.int8)
submission = submission[[ID_COL, TARGET]]

output_path = OUTPUT_DIR / "submission.csv"
submission.to_csv(output_path, index=False)

print(f"Wrote {output_path}")
print(f"submission shape: {submission.shape}")
print(f"predicted positive rate: {submission[TARGET].mean():.5f}")
display(submission.head())


## 7. Next Experiment Notes

The next model should add a ground-contact branch and richer tracking features. Candidate features include nearest-player distance, relative speed, acceleration, orientation alignment, distance changes across adjacent steps, and temporal smoothing by play/pair.
